# 07 — The draft letter, and whether it is complete

The last piece. Feed the **verified** decisions into a letter a human can read, then measure the
only thing worth measuring about it.

---

**Not medical or legal advice. Not a clinical decision tool. Uses public Medicare policy and
synthetic data only. Coverage rules change — always check the current LCD. Outputs may be wrong
and must be reviewed by a qualified human.**

**A human reviews and sends every letter. This project files nothing. Nothing here is submitted
anywhere, and the drafts below are produced from invented patients.**

---

### The metric is completeness, not writing quality

An appeal letter can read beautifully and still be useless, because the reviewer needs specific
administrative facts: the HCPCS code, the AHI, the event count, the adherence figures, what is
actually being requested, and the specific rule being relied on. Published work on LLM-drafted
appeals found exactly that failure — fluent prose, missing numbers.

So this notebook scores a checklist, not style:

| fact | why the reviewer needs it |
|---|---|
| HCPCS code | identifies the item being claimed |
| AHI / RDI | the severity threshold the policy turns on |
| event count | the policy sets a minimum, not just a rate |
| adherence figures | hours per night, % of nights, window length |
| what is requested | initial trial vs continued coverage |
| the rule relied on | a verbatim quote from the policy |

**Only facts the record actually contains are required.** A case where the sleep study never
states an AHI cannot be penalised for omitting one — otherwise the *insufficient* cases would
score badly for the wrong reason.

Two drafters are compared: a **template**, which should score at or near 100% by construction and
acts as the floor, and the **model**, which is the thing actually being measured.

## Setup

In [1]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)

working from: /home/andrew/coding/pa-appeal


### Where this runs

Nothing in this notebook needs a GPU, an embedding model, or Colab. The default path is
standard-library Python over files already in `data/`, so it runs the same on a laptop as in a
hosted runtime.

Only two things are optional extras, and each is needed for exactly one cell.

In [2]:
import importlib.util, sys

def _have(mod):
    return importlib.util.find_spec(mod) is not None

print("python:", sys.version.split()[0])
print("working dir:", Path.cwd())
print()
for mod, why, install in [
    ("google.genai", "only for DRAFT_WITH_MODEL = True", "pip install google-genai"),
    ("gradio",       "only for LAUNCH_DEMO = True",      "pip install gradio"),
]:
    print(f"  {mod:<14} {'present' if _have(mod.split('.')[0]) else 'not installed':<14} {why}")
    if not _have(mod.split(".")[0]):
        print(f"  {'':<14} {'':<14} -> {install}")

print("\nThe template drafter and the completeness scoring need neither of these.")

python: 3.10.20
working dir: /home/andrew/coding/pa-appeal

  google.genai   present        only for DRAFT_WITH_MODEL = True
  gradio         present        only for LAUNCH_DEMO = True

The template drafter and the completeness scoring need neither of these.


In [3]:
cases    = {c["id"]: c for c in json.load(open("data/cases.json"))}
criteria = {c["id"]: c for c in json.load(open("data/criteria.json"))}

RUN_NAME = "full"
rows = json.load(open(f"data/results/{RUN_NAME}/row5_full.json"))["rows"]

by_case = {}
for r in rows:
    by_case.setdefault(r["case_id"], []).append(r)

print(f"{len(cases)} cases, {len(rows)} decisions from row5_full ({RUN_NAME})")

40 cases, 239 decisions from row5_full (full)


## 1. Which decisions belong in the letter

An appeal argues the patient **does** qualify, so the body of the letter is built from decisions
that are both `met` **and** carry a quote that passed verification. That second condition is the
whole point of the project: nothing reaches the letter on the strength of an unverified citation.

The gaps matter too. Criteria that came back `insufficient_evidence` are listed separately, as a
checklist of what the practice still needs to supply. That is more useful to a human than quietly
dropping them, and it is honest about what the record does not show.

In [4]:
def supporting(case_id):
    """met decisions whose quote actually verified against the policy."""
    return [r for r in by_case[case_id]
            if r["predicted_final"] == "met" and r["quote_status"] == "supported"]

def gaps(case_id):
    """criteria the record does not establish either way."""
    return [r for r in by_case[case_id] if r["predicted_final"] == "insufficient_evidence"]

def unverified(case_id):
    """met/unmet decisions whose citation did NOT verify -- deliberately excluded."""
    return [r for r in by_case[case_id]
            if r["predicted_final"] in ("met", "unmet") and r["quote_status"] != "supported"]

demo_id = "case_001"
print(f"{demo_id}: {len(supporting(demo_id))} verified supporting, "
      f"{len(gaps(demo_id))} gaps, {len(unverified(demo_id))} excluded as unverified")

case_001: 5 verified supporting, 0 gaps, 0 excluded as unverified


## 2. The template

Deliberately plain. The letter is a container for facts, and every number in it comes from the
case record rather than from the model — so the template cannot invent anything.

Note the header: every draft says what it is.

In [5]:
DEVICE_NAMES = {"E0601": "single-level continuous positive airway pressure (CPAP) device",
                "E0470": "bi-level device without backup rate",
                "E0471": "bi-level device with backup rate"}

def requested_item(case):
    s = case["spec"]
    period = ("continued coverage beyond the initial three-month trial"
              if s["phase"] == "continued" else "initial coverage")
    return f"{period} of {s['device']} ({DEVICE_NAMES[s['device']]})"


def clinical_facts(case):
    """Only the facts the record actually states. None means the record is silent."""
    s = case["spec"]
    out = []
    if s["ahi"] is not None:
        out.append(f"Apnea-hypopnea index: {s['ahi']} events per hour.")
    if s["events"] is not None:
        out.append(f"Total scored respiratory events: {s['events']}.")
    if s["study_hours"] is not None:
        out.append(f"Recording time: {s['study_hours']} hours.")
    if s["symptom"] not in (None, "none"):
        out.append(f"Documented symptom or condition: {s['symptom']}.")
    if s["phase"] == "continued" and None not in (
            s["usage_hours"], s["usage_pct"], s["usage_window_days"]):
        out.append(f"Device adherence: {s['usage_hours']} hours per night on "
                   f"{s['usage_pct']}% of nights over {s['usage_window_days']} consecutive days.")
    return out


def render_letter(case):
    """Build the draft from verified decisions only. No model involved."""
    cid = case["id"]
    lines = [
        "DRAFT APPEAL - FOR REVIEW BY A QUALIFIED HUMAN BEFORE ANY USE",
        "Generated from a synthetic record. Not medical or legal advice. Nothing is filed.",
        "",
        "Re: Request for redetermination",
        f"Item in dispute: {requested_item(case)}",
        f"Primary diagnosis: {case['spec']['primary_dx'] or 'not stated in the record'}",
        "",
        "We are requesting redetermination of the denial issued for the item above. The",
        "record supports coverage under the applicable Local Coverage Determination",
        "(L33718) for the reasons set out below.",
        "",
        "CLINICAL FACTS IN THE RECORD",
    ]
    facts = clinical_facts(case)
    lines += [f"  - {f}" for f in facts] if facts else ["  - (none stated in the record)"]

    lines += ["", "POLICY CRITERIA THE RECORD SATISFIES"]
    sup = supporting(cid)
    if sup:
        for r in sup:
            crit = criteria[r["criterion_id"]]
            # the canonical sentence from criteria.json, not the model's copy of it:
            # same text, but guaranteed character-for-character against the policy
            lines += [f"  - {r['criterion_id']}: {crit['summary']}",
                      f"    Policy text: \"{crit['text_core']}\""]
    else:
        lines.append("  - (no criterion was established with a verified policy citation)")

    g = gaps(cid)
    if g:
        lines += ["", "NOT YET ESTABLISHED BY THE RECORD",
                  "  The following could not be determined from the documentation supplied.",
                  "  Additional records may resolve them."]
        lines += [f"  - {r['criterion_id']}: {criteria[r['criterion_id']]['summary']}" for r in g]

    lines += ["", "We ask that the denial be reconsidered on the basis of the documentation",
              "cited above.", "", "Enclosures: sleep study report, chart notes, denial notice"]
    return "\n".join(lines)


print(render_letter(cases[demo_id]))

DRAFT APPEAL - FOR REVIEW BY A QUALIFIED HUMAN BEFORE ANY USE
Generated from a synthetic record. Not medical or legal advice. Nothing is filed.

Re: Request for redetermination
Item in dispute: initial coverage of E0601 (single-level continuous positive airway pressure (CPAP) device)
Primary diagnosis: OSA

We are requesting redetermination of the denial issued for the item above. The
record supports coverage under the applicable Local Coverage Determination
(L33718) for the reasons set out below.

CLINICAL FACTS IN THE RECORD
  - Apnea-hypopnea index: 26.5 events per hour.
  - Total scored respiratory events: 137.
  - Recording time: 6.5 hours.
  - Documented symptom or condition: history of stroke.

POLICY CRITERIA THE RECORD SATISFIES
  - initial_evaluation: An in-person clinical evaluation must occur before the sleep test
    Policy text: "The beneficiary has an in-person clinical evaluation by the treating practitioner prior to the sleep test to assess the beneficiary for obstruct

## 3. Completeness

For each case, work out which facts the record *can* support, then check whether the draft
actually contains them.

The check is a substring match on the value as it would be written. That is crude — a bare number
could in principle appear by coincidence — but it is objective and it is the same kind of check the
quote verifier uses, which keeps the whole project honest about what it is measuring.

In [6]:
# Each fact maps to a LIST of acceptable strings, and any one of them counts.
#
# That list is not padding. The first version of this check tested "requested item" by
# looking for the single word "redetermination". Every model letter opened with "I am
# writing to formally appeal the denial of coverage for HCPCS code E0601..." -- it states
# the request perfectly well, just not with that Medicare term of art, which the drafting
# prompt never asked for. Scoring it as a miss reported the model at 78.1% and 0/40
# complete letters; accepting any phrasing of the request gives 97.1% and 34/40.
#
# A checklist that penalises wording rather than content measures the checklist.

def required_facts(case):
    """-> {label: [strings, any of which count]}, limited to facts the record states."""
    s = case["spec"]
    need = {"hcpcs code": [s["device"]],
            "requested item": ["redetermination", "appeal", "reconsider", "request"]}
    if s["ahi"] is not None:
        need["AHI"] = [str(s["ahi"])]
    if s["events"] is not None:
        need["event count"] = [str(s["events"])]
    if s["phase"] == "continued" and None not in (
            s["usage_hours"], s["usage_pct"], s["usage_window_days"]):
        need["adherence hours"] = [str(s["usage_hours"])]
        need["adherence % of nights"] = [str(s["usage_pct"])]
    sup = supporting(case["id"])
    if sup:
        # the quote is checked case-insensitively too: a letter that sentence-cases a
        # quotation has still reproduced it, and the verbatim guarantee is enforced
        # separately against the policy text further down
        need["verbatim policy quote"] = [criteria[sup[0]["criterion_id"]]["text_core"][:60]]
    return need


def completeness(letter, case):
    low = letter.lower()
    need = required_facts(case)
    hit = {k: any(v.lower() in low for v in vs) for k, vs in need.items()}
    return hit, sum(hit.values()), len(hit)


hit, got, want = completeness(render_letter(cases[demo_id]), cases[demo_id])
print(f"{demo_id}: {got}/{want} required facts present\n")
for fact, present in hit.items():
    print(f"  {'yes' if present else 'NO '}  {fact}")

case_001: 5/5 required facts present

  yes  hcpcs code
  yes  requested item
  yes  AHI
  yes  event count
  yes  verbatim policy quote


### Score every case

In [7]:
from collections import Counter

template_letters = {cid: render_letter(c) for cid, c in cases.items()}

missing = Counter()
got = want = 0
per_case = {}
for cid, c in cases.items():
    hit, g, w = completeness(template_letters[cid], c)
    got, want = got + g, want + w
    per_case[cid] = (g, w)
    for fact, present in hit.items():
        if not present:
            missing[fact] += 1

print(f"template drafter: {got}/{want} required facts present "
      f"({got/want:.1%}) across {len(cases)} letters")
print("cases fully complete:", sum(g == w for g, w in per_case.values()), f"/ {len(cases)}")
print("\nmissing facts:", dict(missing) if missing else "none")
print("\nA template scoring ~100% is not a result -- it is the floor. It is here so the")
print("model's number below has something to be compared against.")

template drafter: 210/210 required facts present (100.0%) across 40 letters
cases fully complete: 40 / 40

missing facts: none

A template scoring ~100% is not a result -- it is the floor. It is here so the
model's number below has something to be compared against.


### The property that matters

Every policy sentence in every draft should be reproducible in the source document. This is the
project's central claim applied to the deliverable, so it is worth asserting rather than assuming.

Two things are checked: that no quote which failed verification reached a letter, and that every
quote that did is verbatim policy text under the same normalisation the verifier uses.

In [8]:
import unicodedata

_SUBS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
         "\u2013": "-", "\u2014": "-", "\u00a0": " ", "\u2265": ">=", "\u2264": "<="}

def _norm(t):
    t = unicodedata.normalize("NFKC", t or "")
    for bad, good in _SUBS.items():
        t = t.replace(bad, good)
    return re.sub(r"\s+", " ", t).strip().casefold()

policies = {p.stem: _norm(p.read_text()) for p in Path("data/policies").glob("*.md")}

quoted = leaked = 0
for cid, letter in template_letters.items():
    for r in supporting(cid):
        canonical = criteria[r["criterion_id"]]["text_core"]
        if canonical in letter:
            quoted += 1
            assert any(_norm(canonical) in doc for doc in policies.values()), \
                f"{cid}/{r['criterion_id']}: quote not found in any policy"
    for r in unverified(cid):
        q = r["evidence_quote"].strip()
        if q and q in letter:
            leaked += 1

print(f"policy sentences quoted across {len(template_letters)} letters: {quoted}")
print(f"  all verbatim in the policy corpus : yes (asserted above)")
print(f"  unverified citations that leaked  : {leaked}")
assert leaked == 0, "a letter contains a citation that failed verification"
print("\nA letter can still be wrong -- it inherits the pipeline's label errors. What it")
print("cannot do is show a human a policy sentence that does not exist.")

policy sentences quoted across 40 letters: 181
  all verbatim in the policy corpus : yes (asserted above)
  unverified citations that leaked  : 0

A letter can still be wrong -- it inherits the pipeline's label errors. What it
cannot do is show a human a policy sentence that does not exist.


## 4. The same letter, drafted by the model

Now the interesting one. The model is given exactly the same material the template had — the
verified decisions and the record facts — and asked to write the letter itself.

If the published finding holds, it will read better and contain fewer of the required numbers.

`DRAFT_WITH_MODEL = False` by default. Setting it True costs one call per case (40 for the full
set), cached by prompt like everything else.

In [9]:
import hashlib, random, time

CACHE = Path("data/cache"); CACHE.mkdir(parents=True, exist_ok=True)

# Only what drafting a letter needs. The decision pipeline's Decision/DecisionBatch
# schema lives in notebooks 03 and 04 -- it is not used here, and carrying it would make
# this notebook depend on pydantic for no reason.
#
# The cache key is unchanged from notebook 04, so entries written by either notebook are
# interchangeable. Letter prompts use a different system prompt, so they simply never
# collide with decision prompts.
MODEL = "gemini-3.1-flash-lite"


def ask(prompt, system="", model=MODEL, force=False, response_schema=None):
    """Ask the model once per unique prompt. Repeats come off disk."""
    schema_key = json.dumps(response_schema, sort_keys=True) if response_schema else ""
    key = hashlib.sha256(
        f"{model}|{system}|{schema_key}|{prompt}".encode()).hexdigest()[:16]
    f = CACHE / f"{key}.json"
    if f.exists() and not force:
        return json.loads(f.read_text())["response"]

    try:
        from google import genai
    except ImportError:
        raise ImportError(
            "google-genai is not installed in this environment.\n"
            "  conda:  pip install google-genai\n"
            "  Colab:  %pip install google-genai\n"
            "Only the DRAFT_WITH_MODEL path needs it -- everything else in this "
            "notebook runs without it.") from None

    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    for attempt in range(5):
        try:
            config = {"system_instruction": system, "temperature": 0,
                      "response_mime_type": "application/json"}
            if response_schema is not None:
                config["response_json_schema"] = response_schema
            r = client.models.generate_content(model=model, contents=prompt, config=config)
            text = r.text
            break
        except Exception as e:
            message = str(e).lower()
            retryable = any(t in message for t in
                            ("429", "503", "resource_exhausted", "unavailable"))
            if not retryable or attempt == 4:
                raise
            wait = 5 * (2 ** attempt) + random.random()
            print(f"temporary API error; retrying in {wait:.1f}s")
            time.sleep(wait)

    f.write_text(json.dumps({"model": model, "system": system,
                             "prompt": prompt, "response": text}, indent=2))
    return text

In [10]:

from getpass import getpass
import os

os.environ["GEMINI_API_KEY"] = getpass("Paste the NEW Gemini API key: ")

assert os.environ["GEMINI_API_KEY"], "No API key entered"
print("New API key loaded")


New API key loaded


In [11]:
DRAFT_WITH_MODEL = True     # True = one API call per case
LETTER_LIMIT     = 40        # lower it to try a handful first

LETTER_SYSTEM = """You draft Medicare appeal letters for a human to review and send.

Rules:
1. Use ONLY the facts and policy quotations supplied. Never invent a number, a date, or a
   policy sentence.
2. The letter must be complete enough for a claims reviewer: state the HCPCS code, the
   clinical measurements given, what is being requested, and quote the policy criteria.
3. Quote policy text character-for-character as supplied.
4. Plain professional prose. No letterhead, no signature block.

Return JSON: {"letter": "..."}"""

LETTER_SCHEMA = {"type": "object",
                 "properties": {"letter": {"type": "string"}},
                 "required": ["letter"]}


def model_letter(case):
    cid = case["id"]
    facts = "\n".join(f"- {f}" for f in clinical_facts(case)) or "- (none stated)"
    cited = "\n".join(
        f'- {r["criterion_id"]} ({criteria[r["criterion_id"]]["summary"]}): '
        f'"{criteria[r["criterion_id"]]["text_core"]}"'
        for r in supporting(cid)) or "- (none verified)"
    prompt = (f"REQUESTED ITEM:\n{requested_item(case)}\n\n"
              f"CLINICAL FACTS FROM THE RECORD:\n{facts}\n\n"
              f"POLICY CRITERIA THE RECORD SATISFIES, WITH VERIFIED QUOTATIONS:\n{cited}\n\n"
              f"Draft the appeal letter.")
    return json.loads(ask(prompt, LETTER_SYSTEM, response_schema=LETTER_SCHEMA))["letter"]


if DRAFT_WITH_MODEL:
    from tqdm.auto import tqdm
    targets = list(cases)[:LETTER_LIMIT]
    model_letters = {cid: model_letter(cases[cid]) for cid in tqdm(targets, desc="drafting")}
    print(f"drafted {len(model_letters)} letters")
else:
    model_letters = {}
    print("skipped -- set DRAFT_WITH_MODEL = True to draft with the model")

drafting:   0%|          | 0/40 [00:00<?, ?it/s]

drafted 40 letters


In [12]:
if model_letters:
    m_missing = Counter()
    m_got = m_want = 0
    for cid, letter in model_letters.items():
        hit, g, w = completeness(letter, cases[cid])
        m_got, m_want = m_got + g, m_want + w
        for fact, present in hit.items():
            if not present:
                m_missing[fact] += 1

    t_got = sum(per_case[cid][0] for cid in model_letters)
    t_want = sum(per_case[cid][1] for cid in model_letters)

    print(f"{'drafter':<12} {'facts present':>16} {'complete letters':>18}")
    print("-" * 50)
    print(f"{'template':<12} {t_got:>7}/{t_want:<7} {t_got/t_want:>5.0%} "
          f"{sum(per_case[c][0] == per_case[c][1] for c in model_letters):>10}"
          f"/{len(model_letters)}")
    complete_m = sum(completeness(l, cases[c])[1] == completeness(l, cases[c])[2]
                     for c, l in model_letters.items())
    print(f"{'model':<12} {m_got:>7}/{m_want:<7} {m_got/m_want:>5.0%} "
          f"{complete_m:>10}/{len(model_letters)}")

    print("\nfacts the model left out most often:")
    for fact, k in m_missing.most_common():
        print(f"  {fact:<26} {k}")

    print("\nsample of a model draft:\n")
    print(list(model_letters.values())[0][:700])
else:
    print("no model letters to score")

drafter         facts present   complete letters
--------------------------------------------------
template         210/210      100%         40/40
model            204/210       97%         34/40

facts the model left out most often:
  verbatim policy quote      6

sample of a model draft:

I am writing to formally appeal the denial of coverage for HCPCS code E0601, a single-level continuous positive airway pressure (CPAP) device. The patient's clinical records confirm that all Medicare coverage criteria have been met. The patient has a documented history of stroke and underwent a sleep test that recorded an apnea-hypopnea index of 26.5 events per hour, with 137 total scored respiratory events over a 6.5-hour recording time. This satisfies the requirement that "The apnea-hypopnea index (AHI) or Respiratory Disturbance Index (RDI) is greater than or equal to 15 events per hour with a minimum of 30 events." Furthermore, the patient completed an in-person clinical evaluation prior


## 5. Optional — a paste-in demo

PLAN.md listed this as *if I have time*. It needs `pip install gradio`, which is not in
`requirements.txt`, and it launches a public tunnel when `share=True` — so it is off by default.

It runs the **template** drafter only. Wiring the live pipeline to arbitrary pasted text would
need retrieval and a model call per submission, which is a different and more expensive thing.

In [13]:
LAUNCH_DEMO = True      # pip install gradio first

if LAUNCH_DEMO and not _have("gradio"):
    print("gradio is not installed -- pip install gradio")
elif LAUNCH_DEMO:
    import gradio as gr

    def draft_for(case_id):
        case = cases[case_id]
        letter = render_letter(case)
        hit, got, want = completeness(letter, case)
        table = "\n".join(f"{'yes' if v else 'NO '}  {k}" for k, v in hit.items())
        return letter, f"{got}/{want} required facts\n\n{table}"

    gr.Interface(fn=draft_for,
                 inputs=gr.Dropdown(sorted(cases), label="synthetic case"),
                 outputs=[gr.Textbox(label="draft appeal", lines=28),
                          gr.Textbox(label="completeness", lines=10)],
                 title="CPAP appeal draft (synthetic data only)",
                 description="Drafts are for human review. Nothing is filed."
                 ).launch(share=True)
else:
    print("demo not launched")

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://80d8d69f03f1a61d06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## What this shows

The template exists to be the floor, not the finding. What is worth reporting is the **gap**
between it and the model on the same checklist, over the same cases.

Two things to keep in mind when reading whatever number comes out:

- **Completeness is not correctness.** A letter can contain every required number and still argue
  badly. Nothing here scores the argument.
- **The letter can only be as good as the decisions behind it.** It is built from `row5_full`, so
  it inherits that config's errors — and by construction it silently drops any decision whose
  citation failed verification. That is the intended behaviour, but it means a letter can be
  complete and still be thin because the pipeline could not verify much.

The honest one-line summary of this notebook: *the drafting was never the hard part.*